
# Evaluación Sumativa Unidad 02

COMPONENTE 1: EL DOCUMENTO TÉCNICO (.ipynb)
PARTE 1: Pruebas de Hipótesis Unimuestrales (Tarea APE 09)

# 1. Pruebas de Hipótesis Unimuestrales: Rendimiento Histórico vs. Actual

## Contexto y Parámetro Crítico
Evaluaremos el comportamiento de los precios de las acciones en el mercado bursátil regional. Históricamente, se ha sostenido que el precio promedio de cierre de una acción representativa del sector financiero local ha sido de $\mu_0 = \$2.50$. Queremos investigar si las condiciones económicas recientes han provocado una variación estadísticamente significativa en este valor promedio.

* **Variable de análisis:** Precio de cierre diario de la acción ($X$).
* **Parámetro crítico:** La media poblacional ($\mu$) del precio de cierre.

## Formalismo Matemático y Planteamiento de Hipótesis
Definimos una prueba de hipótesis paramétrica bilateral (dos colas):

$$H_0: \mu = 2.50$$
> *Hipótesis Nula ($H_0$): El precio promedio actual de la acción se mantiene idéntico al valor histórico de \$2.50.*

$$H_1: \mu \neq 2.50$$
> *Hipótesis Alternativa ($H_1$): El precio promedio actual de la acción ha cambiado y es diferente de \$2.50.*

### Estadístico de Prueba
Dado que la varianza de la población total es desconocida y se estimará a partir de la muestra del dataset, el estadístico de prueba riguroso es la $t$ de Student para una muestra:

$$t = \frac{\bar{x} - \mu_0}{\frac{s}{\sqrt{n}}}$$

Donde:
* $\bar{x}$ es la media de la muestra.
* $s$ es la desviación estándar muestral.
* $n$ es el tamaño de la muestra (número de días de negociación registrados).
* El estadístico sigue una distribución $t$ con $\nu = n - 1$ grados de libertad.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# Fijar semilla para reproducibilidad matemática
np.random.seed(42)

# --- SIMULACIÓN DE DATOS DEL DATASET BURSÁTIL ---
# Generamos 45 registros de precios diarios de cierre para la acción en la región
precios_accion = np.random.normal(loc=2.62, scale=0.32, size=45)

# Parámetro bajo la hipótesis nula
mu_0 = 2.50
alpha = 0.05

# --- EJECUCIÓN DEL TEST PARAMÉTRICO ---
# Abstracción avanzada utilizando scipy.stats para una muestra
t_stat, p_val_unimuestral = stats.ttest_1samp(precios_accion, popmean=mu_0)

# --- MÁSCARA DE SALIDA FORMAL ---
print("==================================================")
print("   RESULTADOS: PRUEBA T DE STUDENT UNIMUESTRAL   ")
print("==================================================")
print(f"Tamaño de la Muestra (n)     : {len(precios_accion)} días")
print(f"Media Muestral OBTENIDA (x̄)  : ${np.mean(precios_accion):.2f}")
print(f"Desviación Estándar Muestral(s): ${np.std(precios_accion, ddof=1):.2f}")
print(f"Estadístico T Calculado      : {t_stat:.4f}")
print(f"Valor-p (p-value) obtenido   : {p_val_unimuestral:.4e}")
print("--------------------------------------------------")

# Justificación algorítmica de la decisión basada en el Valor-p
if p_val_unimuestral < alpha:
    print(f"DECISIÓN: Valor-p ({p_val_unimuestral:.4e}) <= alpha ({alpha}). SE RECHAZA H0.")
    print("CONCLUSIÓN: Existe evidencia científica suficiente para afirmar que el precio")
    print("promedio actual de la acción difiere significativamente del histórico de $2.50.")
else:
    print(f"DECISIÓN: Valor-p ({p_val_unimuestral:.4e}) > alpha ({alpha}). NO SE RECHAZA H0.")
    print("CONCLUSIÓN: No existe evidencia estadística para rechazar la estabilidad del precio.")
print("==================================================")

   RESULTADOS: PRUEBA T DE STUDENT UNIMUESTRAL   
Tamaño de la Muestra (n)     : 45 días
Media Muestral OBTENIDA (x̄)  : $2.55
Desviación Estándar Muestral(s): $0.30
Estadístico T Calculado      : 1.1448
Valor-p (p-value) obtenido   : 2.5848e-01
--------------------------------------------------
DECISIÓN: Valor-p (2.5848e-01) > alpha (0.05). NO SE RECHAZA H0.
CONCLUSIÓN: No existe evidencia estadística para rechazar la estabilidad del precio.


# **PARTE 2: Comparación de Grupos (Tareas APE 10 y 11)**

# 2. Comparación de Grupos: Análisis Sectorial en la Bolsa de Valores

## Contexto del Experimento
Para evaluar el comportamiento del mercado bursátil regional de forma multifactorial, dividiremos nuestro conjunto de datos en subgrupos basados en el **Sector Económico** al que pertenecen las acciones negociadas: **Sector Financiero**, **Sector Industrial** y **Sector Comercial**.

Analizaremos si el rendimiento porcentual promedio diario varía de acuerdo al sector empleando un **Análisis de Varianza (ANOVA de 1 factor)** y su respectiva prueba complementaria **Post-Hoc de Tukey**.

## Formalismo Matemático (ANOVA)
Planteamos el contraste de hipótesis para $k = 3$ grupos independientes:

$$H_0: \mu_{\text{Financiero}} = \mu_{\text{Industrial}} = \mu_{\text{Comercial}}$$
> *Hipótesis Nula ($H_0$): No existen diferencias significativas entre los rendimientos promedio diarios de los tres sectores económicos.*

$$H_1: \exists \, (i, j) \quad \text{tal que} \quad \mu_i \neq \mu_j$$
> *Hipótesis Alternativa ($H_1$): Al menos un par de sectores económicos presenta medias de rendimiento diario significativamente distintas.*

El estadístico de contraste $F$ descompone la variabilidad total en variabilidad entre grupos y variabilidad dentro de los grupos (error):

$$F = \frac{MC_{\text{entre}}}{MC_{\text{dentro}}} = \frac{\frac{SC_{\text{entre}}}{k - 1}}{\frac{SC_{\text{dentro}}}{n_T - k}}$$

Donde $SC$ representa las Sumas de Cuadrados, $MC$ las Medias Cuadráticas, $k$ el número de grupos y $n_T$ el tamaño muestral total.

### Supuesto Crítico: Homocedasticidad (Prueba de Levene)
Antes de ejecutar el ANOVA convencional, es un requisito formal indispensable validar que las varianzas de los subgrupos sean homogéneas:
$$H_0: \sigma^2_{\text{Financiero}} = \sigma^2_{\text{Industrial}} = \sigma^2_{\text{Comercial}}$$

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Configuración de simulación de datos muestrales para 3 sectores económicos de Loja
np.random.seed(42)
rend_financiero = np.random.normal(loc=1.35, scale=0.45, size=35) # Sector A
rend_industrial = np.random.normal(loc=0.82, scale=0.50, size=40) # Sector B
rend_comercial  = np.random.normal(loc=1.60, scale=0.42, size=30) # Sector C

# 1. VALIDACIÓN FORMAL DE HOMOCEDASTICIDAD (Test de Levene)
stat_levene, p_val_levene = stats.levene(rend_financiero, rend_industrial, rend_comercial)

print("==================================================")
print("     ANÁLISIS PREVIO: PRUEBA DE HOMOCEDASTICIDAD   ")
print("==================================================")
print(f"Estadístico de Levene : {stat_levene:.4f}")
print(f"Valor-p de Levene     : {p_val_levene:.4f}")

# Evaluación del supuesto para determinar el camino algorítmico riguroso
if p_val_levene > 0.05:
    print("Resultado: Homocedasticidad CONFIRMADA (p > 0.05). Las varianzas son homogéneas.")
    print("Procedemos con el Análisis de Varianza (ANOVA) paramétrico convencional.\n")

    # 2. IMPLEMENTACIÓN AVANZADA DE ANOVA DE 1 FACTOR (scipy.stats)
    f_stat, p_val_anova = stats.f_oneway(rend_financiero, rend_industrial, rend_comercial)

    print("==================================================")
    print("         ANÁLISIS DE VARIANZA (ANOVA 1-FACTOR)     ")
    print("==================================================")
    print(f"Estadístico F de Fisher : {f_stat:.4f}")
    print(f"Valor-p del ANOVA       : {p_val_anova:.4e}")
    print("--------------------------------------------------")

    if p_val_anova < 0.05:
        print(f"DECISIÓN: Valor-p ({p_val_anova:.4e}) <= 0.05. SE RECHAZA H0.")
        print("CONCLUSIÓN: El sector económico influye significativamente en el rendimiento.")
        print("Se requiere ejecutar una prueba Post-Hoc para mapear las diferencias.\n")
        run_post_hoc = True
    else:
        print(f"DECISIÓN: Valor-p ({p_val_anova:.4e}) > 0.05. NO SE RECHAZA H0.")
        print("CONCLUSIÓN: No se detectan diferencias significativas entre los rendimientos sectoriales.")
        run_post_hoc = False
else:
    print("Resultado: HETEROCEDASTICIDAD DETECTADA (p <= 0.05).")
    print("CRÍTICO: No se puede aplicar ANOVA estándar. Se requiere corrección de Welch.")
    f_stat, p_val_anova = stats.alexander_govern(rend_financiero, rend_industrial, rend_comercial)
    print(f"Estadístico Alexander-Govern: {f_stat.statistic:.4f}")
    print(f"Valor-p corregido           : {f_stat.pvalue:.4e}")
    run_post_hoc = False

# PARTE 3: Análisis Post-Hoc de **Tukey**

## Comparaciones Múltiples: Prueba Post-Hoc de Tukey (HSD)
Dado que el ANOVA arrojó un resultado estadísticamente significativo, rechazamos la igualdad global de medias. Para identificar cuáles sectores específicos difieren entre sí sin incrementar la probabilidad de cometer un error de Tipo I (falso positivo), aplicamos el test de **Diferencia Significativa Honesta de Tukey (HSD)**.

El formalismo matemático define el rango estudiantizado ($q$) para contrastar las parejas de medias:

$$q = \frac{\bar{x}_i - \bar{x}_j}{\sqrt{\frac{MC_{\text{dentro}}}{n^*}}}$$

Donde $\bar{x}_i$ y $\bar{x}_j$ representan las medias de los grupos en comparación, y $MC_{\text{dentro}}$ proviene de la varianza residual calculada en el ANOVA.

In [ ]:
# Imports necesarios
import numpy as np
import pandas as pd
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# --- EJEMPLO: descomenta y ajusta estas líneas si necesitas crear datos de prueba ---
# rend_financiero = np.random.normal(loc=0.02, scale=0.01, size=30)
# rend_industrial  = np.random.normal(loc=0.015, scale=0.012, size=28)
# rend_comercial   = np.random.normal(loc=0.018, scale=0.011, size=32)
# ------------------------------------------------------------------------------

# Validar que las variables existen; si no, lanzar error con mensaje útil
required_vars = ['rend_financiero', 'rend_industrial', 'rend_comercial']
missing = [v for v in required_vars if v not in globals() and v not in locals()]
if missing:
    raise NameError(f"Faltan las siguientes variables: {missing}. Definelas antes de ejecutar este bloque.")

# Recuperar las variables (tanto si están en globals() como en locals())
rend_financiero = locals().get('rend_financiero', globals().get('rend_financiero'))
rend_industrial  = locals().get('rend_industrial', globals().get('rend_industrial'))
rend_comercial   = locals().get('rend_comercial', globals().get('rend_comercial'))

# Convertir a arrays y validar que tienen datos
rend_financiero = np.asarray(rend_financiero)
rend_industrial = np.asarray(rend_industrial)
rend_comercial  = np.asarray(rend_comercial)

if rend_financiero.size == 0 or rend_industrial.size == 0 or rend_comercial.size == 0:
    raise ValueError("Una de las muestras está vacía. Verifica tus datos.")

# Decide si ejecutar post-hoc; aquí lo defines manualmente o lo calculas desde el ANOVA
# Ejemplo: run_post_hoc = (pvalue_anova < 0.05)
run_post_hoc = True  # cambia según tu criterio o resultado del ANOVA
alpha = 0.05

if run_post_hoc:
    df_bursatil = pd.DataFrame({
        'Rendimiento': np.concatenate([rend_financiero, rend_industrial, rend_comercial]),
        'Sector': (['Financiero'] * len(rend_financiero) +
                   ['Industrial'] * len(rend_industrial) +
                   ['Comercial'] * len(rend_comercial))
    })

    print("==================================================")
    print("         ANÁLISIS POST-HOC: PRUEBA DE TUKEY (HSD) ")
    print("==================================================")

    tukey_analysis = pairwise_tukeyhsd(
        endog=df_bursatil['Rendimiento'],
        groups=df_bursatil['Sector'],
        alpha=alpha
    )

    print(tukey_analysis.summary())
    print("==================================================")
else:
    print("Prueba Post-Hoc omitida de acuerdo a la decisión de los contrastes previos.")


NameError: Faltan las siguientes variables: ['rend_financiero', 'rend_industrial', 'rend_comercial']. Definelas antes de ejecutar este bloque.